# Best Practices for Prompting in Translation

This Jupyter Python notebook complements chapter 12 of the LT-LiDER book, entitled "Best practices for prompting: how can translation students, trainers and professionals make the most out of LLMs?". A link to the chapter will be provided here as soon as it's published.

To run this notebook first make a copy. You may either (i) copy it to your Google Drive, and then open it there (it will open in the Google Colab environment), which is the recommended option, or (ii) download it and open it in your own computer (you will need a local installation of Jupyter notebooks).

# 1. Provider & Model Selection

Here you need to choose in which mode you want to run the prompts:

1.   **GUI mode** (default). The notebook will generate prompts so you can then copy-paste them into the web UI of your chosen AI provider. To use this mode, set `USE_GUI_MODE=True`.
2.   **API mode**. The notebook will generate the prompts and will send them directly to the model via its API. To use this mode, set `USE_GUI_MODE=False` and specify an LLM provider in the variable `SELECTED_PROVIDER`. Optionally you can select a specific model for that provider in `MODEL_MAP`. When you run the following code cell, you will be asked to enter your API key for the selected provider.

> **Note**: If you want to run the notebook in API mode but do not have an API key, keep in mind that most AI providers require paid billing or credits to use their APIs. An exception is [OpenRouter](https://openrouter.ai/), where you can obtain a free API key.

In [1]:
# Cell 1: User preferences (please modify as you prefer)
import os
import getpass

# Toggle GUI Mode (Set True if you want to copy-paste prompts into web UIs manually. Otherwise the notebook will call the provider's API)
USE_GUI_MODE = True

# LLM provider (only if USE_GUI_MODE=False)
# Supported choices: "openrouter", "gemini", "openai" (chatgpt), "anthropic" (claude), "huggingface", "mistral", "dashscope" (qwen)
SELECTED_PROVIDER = "openai" #"openrouter"

# Set API Keys here or in your environment variables for the selected provider
if not USE_GUI_MODE:
  os.environ[SELECTED_PROVIDER.upper()+"_API_KEY"] = getpass.getpass("Enter API Key: ")

# Change the model of the selected provider (optional, only if USE_GUI_MODE=False)
# for a list of available models check https://models.litellm.ai/
MODEL_MAP = {
    "openrouter": "openrouter/nvidia/nemotron-3.5-lightning:free", #"openrouter/nvidia/nemotron-3-ultra-550b-a55b:free", # see other free models from OpenRouter here: https://openrouter.ai/collections/free-models
    "GEMINI": "gemini/gemini-2.5-flash",
    "OPENAI": "gpt-5.6-luna",
    "ANTHROPIC": "claude-3-5-sonnet-20240620",
    "HUGGINGFACE": "huggingface/meta-llama/Llama-3.1-70B-Instruct",
    "MISTRAL": "mistral/mistral-large-latest",
    "DASHSCOPE": "qwen/qwen-max"
}

# 2. Install required packages, load them and create function to call LLMs

In [2]:
# Cell 2.1: Environment Setup & Package Dependencies
!pip install -q litellm ipywidgets requests pandas

In [3]:
# Cell 2.2: API & GUI Setup
import pandas as pd
from litellm import completion


def generate_response(prompt: str, system_prompt: str = None, temperature: float = 0.3) -> str:
    """
    Executes an API call using LiteLLM across the selected provider or prints
    the formatted prompt for online GUI copy-pasting.
    """
    if USE_GUI_MODE:
        formatted = "=== GUI COPY-PASTE PROMPT ===\n"
        if system_prompt:
            formatted += f"[SYSTEM PROMPT]\n{system_prompt}\n\n"
        formatted += f"[USER PROMPT]\n{prompt}\n"
        formatted += f"[SETTINGS] Temperature: {temperature}\n"
        formatted += "=============================="
        print(formatted)
        return "[GUI Mode Active: Prompt displayed above for web paste]"

    model_name = MODEL_MAP.get(SELECTED_PROVIDER.lower(), "gpt-4o")
    # print(model_name)
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})

    try:
        response = completion(
            model=model_name,
            messages=messages,
            temperature=temperature
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error executing API call with {model_name}: {str(e)}"

print(f"Setup complete. Active Provider: {SELECTED_PROVIDER} | GUI Mode: {USE_GUI_MODE}")

Setup complete. Active Provider: openai | GUI Mode: True


# 3. Text Input Handler
**Cell 3** provides a helper function that can be used to load text from a string, or from a file.

The text is stored in variable `source_text`, which later on can be used as input to any of the prompts in Sections 4 and 5.

In [4]:
# Cell 3: Input Upload or Direct Copy-Paste Function

import os


def load_input_text(direct_text: str | None = "", file_path: str | None = None) -> str:
    """
    Loads text from a local file path, a direct string entry, or a Colab file upload.
    """
    raw_bytes = None

    # 1. Load from an explicit file path (fail loudly if the path is invalid)
    if file_path:
        if not os.path.exists(file_path):
            raise FileNotFoundError(
                f"File not found: '{file_path}'. Please check the file name and path."
            )
        if file_path.lower().endswith((".docx", ".pdf", ".xlsx")):
            raise ValueError(
                f"Unsupported file format for '{file_path}'. Please provide a plain-text (.txt) file."
            )
        with open(file_path, "rb") as f:
            raw_bytes = f.read()

    # 2. Otherwise, load from direct string input
    elif direct_text and direct_text.strip():
        return direct_text.strip()

    # 3. Otherwise, fall back to the Colab file upload prompt
    else:
        try:
            from google.colab import files
            print("Please select a plain-text (.txt) file to upload:")
            uploaded = files.upload()
            for name, content in uploaded.items():
                if name.lower().endswith((".docx", ".pdf", ".xlsx")):
                    raise ValueError(
                        f"Unsupported file format for '{name}'. Please upload a plain-text (.txt) file."
                    )
                raw_bytes = content
                break
        except ImportError:
            pass

    # Decode bytes (strips UTF-8 BOM if present; falls back to Windows-1252)
    if raw_bytes is not None:
        try:
            return raw_bytes.decode("utf-8-sig").strip()
        except UnicodeDecodeError:
            return raw_bytes.decode("cp1252").strip()

    print("No input provided; falling back to default sample text.")
    return "Sample text: Default translation source input."


# Example Input Initialization
# Option 1 (Default): Enter text directly
source_text = load_input_text(
    direct_text=(
        "Our newly developed foundation highlights your natural beauty. "
        "It melts seamlessly into the skin, providing a finish that looks just like your natural skin."
    )
)

# Option 2: Upload a .txt file interactively in Colab (uncomment to use)
# source_text = load_input_text()

# Option 3: Load from a local/Colab file path (uncomment to use)
# source_text = load_input_text(file_path="my_source_text.txt")

print("Active Source Text:\n", source_text)

Active Source Text:
 Our newly developed foundation highlights your natural beauty. It melts seamlessly into the skin, providing a finish that looks just like your natural skin.




---


# 4. Chapter Examples part 1 (Effective Prompting)

Demonstrating role assignment, parameter variations, zero/few-shot prompts,  Chain-of-Thought, and the CO-STAR framework.

## 4.1 Give clear and specific instructions

Personalising the model (Example 1 in Section 4.1 in the Chapter) might contribute to better results, e.g. “You are a professional technical translator” rather than only “Translate”.

In [ ]:
# Cell 4.1.1: Personalising the model (Example 1)

source_text = "El perro la sigue unos cuantos pasos por detrás como si pretendiera pasar inadvertido."

persona_prompt1 = f"""You are a professional literary translator with a deep understanding of both English and Spanish literature. Translate the following text from Spanish to English, preserving the original tone, style, and emotional nuance. Avoid overly literal translation and aim for natural, expressive phrasing in the target language.

Text: {source_text}"""

print(generate_response(persona_prompt1, temperature=1.0))

The dog follows her a few steps behind, as if trying to go unnoticed.


Sample output with openrouter/nvidia/nemotron-3-ultra-550b-a55b:free (and with gpt-5.6-luna)

```
The dog trails a few steps behind her, as if trying to go unnoticed.
```

With openrouter/nvidia/nemotron-3.5-lightning:free

```
The dog follows a few steps behind her, as if it meant to go unnoticed.
```

In [ ]:
# Cell 4.1.2: Personalising the model (Example 2)
# This example is taken from Yamada (2023). https://aclanthology.org/2023.mtsummit-users.19/

source_text = "私たちが開発したファンデーションはあなたの自然な美しさを引き立てます。シームレスに肌に溶け込み、まるで素肌そのもののような仕上がりを提供します。"

persona_prompt2 = f"""Translate the following Japanese text into English. Please fulfill the following conditions when translating.
Purpose of the translation: To market our own brand of cosmetics and to be displayed on our website
Target audience: Women in their 20s

Text: {source_text}"""

print(generate_response(persona_prompt2, temperature=1.0))

Our foundation enhances your natural beauty, seamlessly melting into the skin for a finish that feels just like bare skin.


Sample output with gpt-5.6-luna (API mode):

```
Our specially developed foundation enhances your natural beauty, effortlessly blending into your skin for a flawless, bare-skin look.
```

<br><br>
Sample output with Gemini 3.6 Flash (GUI mode):


### Option 1: Fresh & Direct (Best for quick scanning/hero banners)

Designed to enhance your natural beauty, our foundation seamlessly melts into your skin for a effortless, second-skin finish.

### Option 2: Engaging & Trendy (Best for Gen Z/Young Millennial appeal)

Reveal your natural glow. Our custom-formulated foundation melts seamlessly into skin, giving you that fresh, "your-skin-but-better" finish.

### Key Translation Adjustments for Your Target Audience:



1.   "まですご素肌そのもののような" (Like real skin itself): Rendered as "second-skin finish" or "your-skin-but-better", which are standard, highly appealing marketing terms for cosmetics aimed at women in their 20s.

2.   "引き立てます" (Enhances/Brings out): Framed around "natural glow" and "enhancing natural beauty" to align with modern, low-maintenance, skin-first beauty trends.

3.   Tone: Kept light, confident, and approachable rather than overly stiff or literal ("Our developed foundation provides...").


## 4.2. Give the model time to process



In [5]:
# Cell 4.2.1: Example 3

source_language = "English"
target_language = "Spanish"
source_text = """Most approved systemic treatments for atopic dermatitis act by suppressing inflammatory effector pathways. Dupilumab blocks interleukin (IL)-4 and IL-13 signalling; tralokinumab and lebrikizumab target IL-13; nemolizumab targets IL-31 receptor alpha; and JAK inhibitors suppress several intracellular cytokine signalling pathways simultaneously. These approaches have transformed care, but they do not directly restore the upstream immune regulatory defects that may contribute to disease persistence. Regulatory T cells are central to peripheral immune tolerance, and abnormalities in regulatory T-cell number or function have been described in allergic and inflammatory diseases, including atopic dermatitis. This provides the rationale for therapies that seek not only to inhibit inflammation but also to re-establish immune regulation.

IL-2 biology is attractive in this regard, but native IL-2 is not a selective regulatory T-cell therapy. At higher or non-selective exposures, IL-2 may also expand natural killer cells and effector T cells, limiting its therapeutic usefulness. 5 Engineering IL-2-based molecules that preferentially signal through the high-affinity IL-2 receptor, which is enriched on regulatory T cells, could allow selective regulatory T-cell expansion while avoiding broader immune activation. Rezpegaldesleukin is a pegylated IL-2 receptor agonist developed with this aim. """

time_prompt = f"""Translate a pharmaceutical text from {source_language} to {target_language}.
First, find the most frequent words used. Extract the source words in a three-column table: one column containing the source word, another one
the definition including a reference link, and the third the translated word into [target language] including a reference link to the translated word.

Text: {source_text}"""

print(generate_response(time_prompt, temperature=0.2))

=== GUI COPY-PASTE PROMPT ===
[USER PROMPT]
Translate a pharmaceutical text from English to Spanish.
First, find the most frequent words used. Extract the source words in a three-column table: one column containing the source word, another one
the definition including a reference link, and the third the translated word into [target language] including a reference link to the translated word.

Text: Most approved systemic treatments for atopic dermatitis act by suppressing inflammatory effector pathways. Dupilumab blocks interleukin (IL)-4 and IL-13 signalling; tralokinumab and lebrikizumab target IL-13; nemolizumab targets IL-31 receptor alpha; and JAK inhibitors suppress several intracellular cytokine signalling pathways simultaneously. These approaches have transformed care, but they do not directly restore the upstream immune regulatory defects that may contribute to disease persistence. Regulatory T cells are central to peripheral immune tolerance, and abnormalities in regulatory T

In [ ]:
# Cell 4.2.2: Example 4

source_language = "Spanish"
target_language = "English"
source_text = "" #"Mi casa está al otro lado del río."

time_prompt = f"""You are a professional translator. Translate the following text from {source_language} to {target_language}. First, let's analyse the text structure, tone,
and any possible ambiguities.

Text: {source_text}"""
print(generate_response(time_prompt, temperature=1.0))

**Text Analysis**

- **Structure:** The sentence follows a standard Spanish SVO (Subject-Verb-Prepositional Phrase) order. 
  - *Mi* = possessive determiner ("my")
  - *casa* = noun ("house")
  - *está* = verb *estar* ("is"), used for fixed or situational location
  - *al otro lado del río* = prepositional phrase = *a el + otro lado + de el + río* = "on the other side of the river"
  The phrasing is grammatically straightforward with no clausal embedding or complex syntax.

- **Tone:** Neutral, descriptive, and factual. There is no evident emotion, irony, or Register shift. It functions like a simple location statement (e.g., giving directions or describing a geographic relationship).

- **Possible Ambiguities:** 
  - *"Mi casa"* could stylistically shift to *"the house"* in a very generic description, but *"my house"* is the default and most natural translation.
  - *"Al otro lado del río"* is spatially literal. Context could theoretically make it metaphorical (e.g., "on the other sid

## 4.3. Temperature

Temperature is a parameter that controls the degree of randomness when selecting the next token. Many LLMs allow temperatures between 0 and 2, although the exact range depends on the model and implementation.

In [ ]:
# Cell 4.3

# Here we go back to example 1 and now call the LLM with 2 different settings of temperature: one very low and one very high.
# Note: the temperature range can differ per model so it's recommended to check in its release notes.

print("--- Persona Output (Low Temp 0.2) ---")
print(generate_response(persona_prompt1, temperature=0.2))

print("\n--- Persona Output (High Temp 1.0) ---")
print(generate_response(persona_prompt1, temperature=1.0))

--- Persona Output (Low Temp 0.2) ---
The dog follows a few steps behind her, as if trying to pass unnoticed.

--- Persona Output (High Temp 1.0) ---
The dog followed a few steps behind her, as if trying to pass unnoticed.


Sample output with gpt-5.6-luna:

*   Temperature 0.2: The dog follows her a few steps behind, as if trying to go unnoticed.

*   Temperature 2.0: The dog trails a few steps behind her as if trying to go unnoticed.





## 4.4. Zero-shot, one-shot and few-shot prompting

Zero-shot means that no examples are given to the model, just the actual prompt, the instruction, e.g. “Translate” or “Review this translation”.

One-shot and few-shot prompting, on the other hand, refer to the fact that
examples are given to the model at the prompting stage so that it can emulate
them in its output, as the name indicates one or more than one, respectively.

In [ ]:
# Cell 4.4.1 One-Shot Prompting (Example 5)
one_shot_prompt = """You are a professional literary translator. Please translate the following French sentence into natural, fluent English, preserving the original tone and nuance.

Example:
French: “Il faisait un froid de canard ce matin.”
English: “It was bitterly cold this morning.”

Now translate this:
French: “Elle a laissé tomber le livre comme si c’était un poids mort.”
English:"""

print(generate_response(one_shot_prompt, temperature=1.0))

She dropped the book as if it were a dead weight.


Sample output with gpt-5.6-luna:

```
“She dropped the book as if it were a dead weight.”
```

In [ ]:
# Cell 4.4.2 Few-Shot Prompting (Example 6)
few_shot_prompt = """You are a professional literary translator. Translate the following Spanish sentences into natural, elegant English, preserving tone, imagery, and emotional nuance.

Examples:
Spanish: "Esta mañana, hacía un frío que pelaba."
English: "It was freezing cold this morning."

Spanish: "El cielo se cubrió de nubes como si alguien lo hubiera pintado con ceniza."
English: "The sky was suddenly covered in clouds, as if someone had painted it with ashes."

Spanish: “Dejó caer el libro como si ya no le importara nada.”
English: “She dropped the book as if nothing mattered anymore.”

Now translate this:
Spanish: "El perro la sigue unos cuantos pasos por detrás como si pretendiera pasar inadvertido."
English:"""
print(generate_response(few_shot_prompt, temperature=1.0))

The dog followed her a few steps behind, as if he meant to go unnoticed.


Sample output with openrouter/nvidia/nemotron-3-ultra-550b-a55b:free

```
The dog trails her by a few paces, as though hoping to pass unseen.
```

Sample output with gpt-5.6-luna

```
The dog follows her a few steps behind, as if trying to go unnoticed.
```

## 4.5. Chain of thought (CoT)

CoT is a prompting technique whereby the model is asked to generate a series of intermediate steps before arriving at an answer.

In [ ]:
# Cell 4.5.1. Chain-of-Thought (CoT) Strategy (Example 7)
cot_prompt = f"""You are a professional literary translator. Translate the following text from English to Spanish. Explain the process by highlighting the translation problems in the source text and how they were resolved.

Text: {source_text}"""

print("\n--- Chain-of-Thought Output ---")
print(generate_response(cot_prompt, temperature=1.0))


--- Chain-of-Thought Output ---
**Note on language direction:**  
The text you provided, *"Mi casa está al otro lado del río,"* is already in Spanish. Assuming the intended English source is *"My house is on the other side of the river,"* I’ll translate it into Spanish and use that as the basis for a literary translation analysis. This approach respects your request while transparently addressing the language mismatch.

### Translation
**Spanish:** *Mi casa está al otro lado del río.*  
(Identical to the provided text, as it is already a natural, idiomatic Spanish sentence.)

### Translation Problems & Resolution Process

If we treat *"My house is on the other side of the river"* as the English source, several discrete translation decisions arise. Here’s how I navigated them:

| English Segment | Literal Spanish Calque | Idiomatic/Literary Choice | Resolution |
|----------------|------------------------|---------------------------|------------|
| **"house"** | *casa* | *casa* | Both a

Sample output with gpt-5.6-luna:

```

--- Chain-of-Thought Output ---
As a professional literary translator, my task is to accurately and fluently convey the meaning and tone of the original text in the target language, while considering cultural nuances and possible translation challenges. Let's examine the provided text, which is originally in Japanese, and translate it into Spanish:

Original Text (Japanese):
私たちが開発したファンデーションはあなたの自然な美しさを引き立てます。シームレスに肌に溶け込み、まるで素肌そのもののような仕上がりを提供します。

The first step in translation is to understand the content and intent of the original text. This text is an advertisement for a cosmetic product—a foundation—that enhances natural beauty and blends seamlessly into the skin for a natural finish. Now, let's address specific translation problems and how they were resolved:

1. **Japanese Syntax and Sentence Structure**:
    - Japanese often places the verb at the end of the sentence, which differs from typical Spanish sentence structure. This requires reordering when translating to maintain grammatical correctness and readability in Spanish.

2. **Cultural Equivalence and Tone**:
    - The text uses the phrase "自然な美しさを引き立てます" (shizen'na utsukushisa o hikitate masu), meaning "enhances your natural beauty." It's important to maintain the promotional tone which is often direct and engaging in Spanish advertising.

3. **Technical Terms (Cosmetic Language)**:
    - Terms like "ファンデーション" (foundation) need to be accurately translated into "base de maquillaje" to ensure the terminology is familiar to Spanish-speaking audiences.

Translation to Spanish:
"La base de maquillaje que hemos desarrollado realza tu belleza natural. Se funde de manera uniforme con la piel, ofreciendo un acabado que parece la piel misma."

Resolution of Issues:
- **Structure Adjustment**: The phrase structure was adjusted to follow Spanish syntax, which typically requires subject-verb-object order.
- **Consistency in Tone**: Ensured the tone remains promotional and engaging by choosing dynamic verbs like "realza" (enhances) and "se funde" (blends).
- **Accurate Terminology**: The term "ファンデーション" was translated to "base de maquillaje", an industry-standard term in Spanish-speaking regions.

In summary, the translation process involved understanding the source text's intent and style, restructuring sentences to fit Spanish language conventions, maintaining brand tone, and using culturally appropriate terminology.
```

In [ ]:
# Cell 4.5.2. CoT combined with few-shot prompting (Example 8)

source_language = ""
target_language = ""
text = ""

cot_fs_prompt = f"""You are a professional literary translator. Translate the following text from {source_language} to {target_language}. Explain the process by highlighting
the translation problems in the source text and how they were resolved.

Example:
{source_language}: [source text example]
{source_language}: [target text example]

Text: {source_text}"""

print("\n--- Chain-of-Thought combined with Few-shot Output ---")
print(generate_response(cot_fs_prompt, temperature=1.0))


--- Chain-of-Thought combined with Few-shot Output ---
### : Spanish → English
### : My house is on the other side of the river.

**Translation Process & Problem-Solving**

1. **Grammatical Contraction (`al`)**  
   *Source:* “al otro lado” (contraction of *a + el*).  
   *Problem:* A literal word-for-word rendering (“to the other side”) would sound clunky in English and miss the natural flow.  
   *Resolution:* I rendered it as “on the other side,” which preserves the spatial meaning while sounding idiomatic in English. The preposition “on” also subtly captures the nuance of *al* when referring to a position alongside or spanning a boundary like a river.

2. **Lexical Choice for `otro lado`**  
   *Source:* *otro lado* literally means “other side.”  
   *Problem:* Depending on context, this could mean “the opposite bank,” “across,” or even “beyond” in a metaphorical sense. A rigid translation (“the other side”) is correct but bland.  
   *Resolution:* I chose “on the other side of th

## 4.6 The CO-STAR Framework
To structure some of the strategies described herein, we would like to offer a consolidated framework that can be used as a guideline: the CO-STAR framework (**C**ontext, **O**bjective, **S**tyle, **T**one, **A**udience, **R**esponse) by [Teo (2024)](https://medium.com/data-science/how-i-won-singapores-gpt-4-prompt-engineering-competition-34c195a93d41).



In [ ]:
# Cell 4.6: CO-STAR Framework Implementation (Example 9)

source_language = "English"
target_language = "Dutch"
text = "Standardize technical documents with consistent styles, headings, and tables. Ensure specifications align across projects and reuse finalized documents to maintain the same structure over time."
customer_style = "Microsoft Style Guide"
target_country = "The Netherlands"

costar_prompt = f"""#Context#
You are a professional technical translator.

#Objective#
Translate the following text from {source_language} to {target_language}.

#Text#
{text}

#Style#
This is a business brief, following the style of {customer_style}.

#Tone#
The tone of the text is informative.

#Audience#
The translation is intended for the top executives of the company {target_country}.

#Response#
First, explain the process by highlighting the translation problems in the source text. Then, provide solutions for these problems. Finally, provide an Excel table with the source in column A, and the target in column B.
"""

print(generate_response(costar_prompt, temperature=0.3))

### Translation Problems Analysis
The source text is concise but contains several elements that require careful handling for a Dutch audience of top executives, especially under Microsoft Style Guide conventions (plain language, active voice, minimal Anglicisms, clear structure).

1. **"Standardize"** – The verb is direct but slightly abstract in Dutch. While "standaardiseren" is correct, Microsoft style favors active, tangible phrasing that immediately signals action to executives.
2. **"consistent styles, headings, and tables"** – A noun cluster ("styles, headings, and tables") can feel dense in Dutch. The guide recommends breaking such clusters or using parallel structure for better readability, especially in briefs for C-level audiences.
3. **"specifications align"** – "Align" is a common Anglicism in tech. The Microsoft Style Guide advises preferring native Dutch equivalents like "overeenkomen" or "harmoniseren" unless the term is fully institutionalized.
4. **"across projects"** 



---


# 5. Chapter Examples part 2 (Prompting for Translation)




## 5.1. Terminology Creation & Review
Extracting domain-specific terms and generating structured bilingual tables with external definitions.

In [ ]:
# Cell 5.1: Terminology Extraction (Example 10)

source_language = "English"
target_language = "Dutch"
source_text = " Most approved systemic treatments for atopic dermatitis act by suppressing inflammatory effector pathways. Dupilumab blocks interleukin (IL)-4 and IL-13 signalling; tralokinumab and lebrikizumab target IL-13; nemolizumab targets IL-31 receptor alpha; and JAK inhibitors suppress several intracellular cytokine signalling pathways simultaneously. "
customer_style_guide = "None"
term_example = "dermatitis"

term_prompt = f"""#Context#
You are a professional translator specialized in medical texts.

#Objective#
Extract the domain-specific terms, abbreviations, named entities and recurring technical expressions from the following text in {source_language} and create a glossary of the important key terms from {source_language} into {target_language}. Prioritise relevance, accuracy, and clarity in the definitions.

#Text#
{source_text}

#Style#
This is a medical text, specialised in {customer_style_guide}.

#Tone#
Informative.

#Audience#
The translation is intended for experts in this domain.

#Response#
Create a markdown table with the following columns:
1. Column A: Source term in English
2. Column B: Concise definition based on an authoritative source
3. Column C: URL reference to an authoritative source where available
4. Column D: Translation into {target_language}
5. Column E: Concise definition in {target_language}
6. Column F: URL reference in {target_language}

If multiple translations are possible, recommend the preferred translation and briefly justify the choice.

Sort the glossary alphabetically by the source term.

#Examples#
This is an example of a key term, definitions, source and translation, defini-
tions and source:
{term_example}
"""

print(generate_response(term_prompt, temperature=0.1))

**Glossary of Key Medical Terms – English to Dutch**  
*Sorted alphabetically by the English source term. Definitions are concise, authoritative, and tailored for expert audiences. Preferred Dutch translations are recommended where multiple exist, with brief justifications embedded in the definitions.*

| A (Source term in English) | B (Concise definition based on authoritative source) | C (URL reference to authoritative source) | D (Translation into Dutch) | E (Concise definition in Dutch) | F (URL reference in Dutch) |
|---|---|---|---|---|---|
| atopic dermatitis | A chronic, relapsing inflammatory skin disease characterized by pruritus, eczematous lesions, and impaired skin barrier, driven by Th2‑mediated immunity. | https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7877735/ | atopische dermatitis | Een chronische, recurrente inflammatoire huidziekte gekenmerkt door jeuk, eczema‑leties en een beschadigd huidbarrièrestelsel, aangedreven door Th2‑gemeden immuunreacties. *(Voorkeursvertali

# 5.2: Style Transformations & Language Variants
Adapting regional varieties, conducting style transfer, and enforcing gender-inclusive templates.

In [ ]:
# Cell 5.2.1. Change of Language Variant (Example 11)

argentinian_rewrite = f"""You are a professional editor. Your task is to rewrite the following Spanish text for an Argentinian audience, paying attention to variance in vocabulary, expressions, idioms, and grammar (voseo). Make sure that the meaning of the original text is not altered in any way.

Text: "Si necesitas ayuda con tu compra, habla con nuestro soporte hoy mismo." """

print("--- Regional Variant Shift ---")
print(generate_response(argentinian_rewrite, temperature=1.0))

--- Regional Variant Shift ---
Si necesitás ayuda con tu compra, hablá con nuestro soporte hoy mismo.


Sample output with Gemini 3.6 Flash (Web): "Si necesitás ayuda con tu compra, hablá con nuestro soporte hoy mismo."

In [ ]:
# Cell 5.2.2. Variant Classification (Example 13)
classification_prompt = """You are a professional reviewer. Given the text below, review the sentence and determine which Spanish variant is used. Output "ES-AR" for Argentinian, "ES-ES" for Spain, or "Both" if ambiguous.

Text: "Tenés que traer el auto al taller mañana por la mañana." """

print("\n--- Language Variant Classification ---")
print(generate_response(classification_prompt, temperature=0.0))


--- Language Variant Classification ---
ES-AR


Sample output (Gemini 3.6 Flash): "ES-AR"

In [ ]:
# Cell 5.2.3. Gender-Specific Template (Example 14, from Sánchez et al. (2024), https://doi.org/10.26615/issn.2683-0078.2023_021)

gender_template_prompt = """Translate the following English sentences into both grammatical genders in Spanish. Provide two Spanish sentences for each English source sentence.

Examples:
English: I have friends who are Hispanic people.
Spanish: Tengo amigos que son personas hispanas.
Spanish: Tengo amigas que son personas hispanas.

English: What do you think of ginger children?
Spanish: ¿Qué piensas de los niños pelirrojos?
Spanish: ¿Qué piensas de las niñas pelirrojas?

English: I have friends who are orphans.
Spanish: Tengo amigos huérfanos.
Spanish: Tengo amigas huérfanas.

English: The customer wants to receive their medical report.
Spanish:"""


print("\n--- Gender-Specific Dual Outputs ---")
print(generate_response(gender_template_prompt, temperature=0.2))



--- Gender-Specific Dual Outputs ---
Spanish: El cliente quiere recibir su informe médico.  
Spanish: La cliente quiere recibir su informe médico.


## 5.3. Prompting for Post-editing

We are offering here two types of prompts varying in complexity, the first one instructing the system to correct errors based on guidelines, the second one including Chain-of-Thought strategies.

In [ ]:
# Cell 5.3.1. Post-editing prompt 1 (Example 15)

source_text = "Make sure the user has read all safety instructions before first use."
target_text = "Zorg ervoor dat de gebruiker vóór het eerste gebruik alle veiligheidsinstructies heeft gelezen."
guidelines = "The final quality of this translation is publishable quality."

ape_prompt1= f"""Compare the English and the Dutch translation produced by the DeepL machine translation system and then correct its output.
Revise the translation according to the following guidelines.
If the machine translation is already correct, do not make any changes.
Preserve all correct translations.

Guidelines: {guidelines}

English: {source_text}

Dutch: {target_text}"""

print("--- Automatic Post-Editing 1---")
print(generate_response(ape_prompt1, temperature=0.2))

--- Automatic Post-Editing 1---
The provided Dutch translation is already accurate, grammatically correct, and of publishable quality. It faithfully reflects the English source:

- **"Make sure"** → *"Zorg ervoor dat"* (correct idiomatic equivalent)  
- **"the user"** → *"de gebruiker"* (correct)  
- **"has read"** → *"heeft gelezen"* (correct present perfect)  
- **"all safety instructions"** → *"alle veiligheidsinstructies"* (correct)  
- **"before first use"** → *"vóór het eerste gebruik"* (correct word order and meaning)

No changes are needed. The translation meets the guidelines and can be published as-is.

**Dutch (unchanged):**  
`Zorg ervoor dat de gebruiker vóór het eerste gebruik alle veiligheidsinstructies heeft gelezen.`


In [ ]:
# Cell 5.3.1. Post-editing prompt 2 (Example 16)

ape_prompt2 = """As an expert translation post-editor, your task is to improve the German translation for the below English text:

English: They were addressed to her son, who has autism and lives in a private care facility, she said. But instead of her son's name inside when you opened them, the letters said Dear Maine’s Department of Health and Human Services –in Cincinnati, she told local media.
German MT: Sie waren an ihren Sohn gerichtet, der Autismus hat und in einer privaten Pflegeeinrichtung lebt, sagte sie. Aber anstelle des Namens ihres Sohnes im Inneren, als Sie sie öffneten, hieß es in den Briefen Dear Maine Dear Maine ’s Department of Health and Human ServicesServices - in Cincinnati, sagte sie den lokalen Medien.

To accomplish this, follow these steps:
Step 1: Say "Proposed Improvements:". Then brainstorm and design the improvements that make the German translation more faithful and fluent.
Step 2: Say "Improved Translation:". Then output the German translation with proposed improvements that increase translation faithfulness and fluency."""

print("--- Automatic Post-Editing 2---")
print(generate_response(ape_prompt2, temperature=0.2))

--- Automatic Post-Editing 2---
Here's a thinking process:

1.  **Analyze the User's Request:**
   - **Role:** Expert translation post-editor
   - **Task:** Improve a German translation of an English text
   - **Input English:** "They were addressed to her son, who has autism and lives in a private care facility, she said. But instead of her son's name inside when you opened them, the letters said Dear Maine’s Department of Health and Human Services –in Cincinnati, she told local media."
   - **Current German MT (Machine Translation):** "Sie waren an ihren Sohn gerichtet, der Autismus hat und in einer privaten Pflegeeinrichtung lebt, sagte sie. Aber anstelle des Namens ihres Sohnes im Inneren, als Sie sie öffneten, hieß es in den Briefen Dear Maine Dear Maine ’s Department of Health and Human ServicesServices - in Cincinnati, sagte sie den lokalen Medien."
   - **Required Steps:**
     - Step 1: Say "Proposed Improvements:", then brainstorm and design improvements for faithfulness and 

Sample output with Gemini 3.6 Flash

Proposed Improvements:

*   Fix Translation Artifacts and Typos: The MT output contains severe repetitive errors ("Dear Maine Dear Maine ’s", "ServicesServices") caused by machine translation glitches that need to be completely cleaned up.

*   Grammar & Pronoun Fix: "als Sie sie öffneten" uses the formal second-person pronoun "Sie" (You). In this narrative context, "you opened them" functions as a generic "one opened them," which translates better to passive voice or "beim Öffnen" (upon opening).

*   Translate Salutation Contextually: Leaving "Dear Maine’s Department..." raw or semi-translated breaks German narrative flow. It should be translated into idiomatic German (e.g., „Sehr geehrtes Gesundheitsministerium des US-Bundesstaates Maine…“) to convey the absurd wrong-address situation described in the news item.

*   Fluency & Word Choice:

    *   Replace the clunky "sagte sie den lokalen Medien" with more natural journalistic phrasing like "berichtete sie lokalen Medien" or "sagte sie gegenüber lokalen Medien."

    *   Refine "hieß es in den Briefen" to flow better with the nested direct/indirect quote structure.

Improved Translation:
Sie waren an ihren Sohn gerichtet, der Autismus hat und in einer privaten Pflegeeinrichtung lebt, sagte sie. Doch anstatt des Namens ihres Sohnes stand beim Öffnen der Briefe als Anrede „Sehr geehrtes Gesundheitsministerium von Maine“ – und das mitten in Cincinnati, wie sie gegenüber lokalen Medien berichtete.

## 5.4. Prompting for Creative Texts

In [ ]:
# Cell 5.4. Creative Translation Prompt (Example 17, from Du et al. (2025), https://aclanthology.org/2025.mtsummit-1.44/)

target_language = "Dutch"
text = "Everything was perfectly swell"

creative_prompt = f"""Translate the following text into {target_language} creatively.

Text: {text}"""


print("\n--- Creative Translation")
print(generate_response(creative_prompt, temperature=1.0))


--- Creative Translation
Alles was top, pure klasse, alsof de wereld even stil stond van pure tevredenheid.


## 5.5. Translation Evaluation & Quality Assurance (LQA)
Evaluating MT or human output against structured criteria and outputting valid JSON or table metrics.

In [ ]:
# Cell 5.5.1. MT Evaluation (Example 18, Qian et al (2024) from Kocmi and Federmann (2023), https://aclanthology.org/2024.emnlp-main.214/ and https://doi.org/10.18653/v1/2023.wmt-1.64)

source_lang = "English"
target_lang = "German"
translation_information = """MT output: Sie waren an ihren Sohn gerichtet, der Autismus hat und in einer privaten Pflegeeinrichtung lebt, sagte sie. Aber anstelle des Namens ihres Sohnes im Inneren, als Sie sie öffneten, hieß es in den Briefen Dear Maine Dear Maine ’s Department of Health and Human ServicesServices - in Cincinnati, sagte sie den lokalen Medien.

Source text: They were addressed to her son, who has autism and lives in a private care facility, she said. But instead of her son's name inside when you opened them, the letters said Dear Maine’s Department of Health and Human Services –in Cincinnati, she told local media.
"""

mt_eval_prompt1 = f"""Score the following translation from {source_lang} to {target_lang} with respect to the source text on a continuous scale from 0 to 100, where a score of zero means “no meaning preserved” and a score of one hundred means “perfect meaning and grammar”.

{translation_information}

Score:"""

print(generate_response(mt_eval_prompt1))


The translation successfully conveys the overall narrative and sentence structure, but contains a significant error in the quoted letter content: the source text "Dear Maine’s Department of Health and Human Services –in Cincinnati" is rendered as "Dear Maine Dear Maine ’s Department of Health and Human ServicesServices - in Cincinnati," with a repeated "Dear Maine," a garbled "ServicesServices," and a slightly awkward possessive/wording. The rest of the German rendering is largely accurate, though "der Autismus hat" is colloquially acceptable and "sagte sie den lokalen Medien" is a valid though slightly literal construction. Because the core meaning is mostly preserved but the exact phrasing of a key segment is corrupted, the score reflects strong overall quality with a notable flaw.

Score: 70


Output with Gemini 3.6 Flash

Score: 55/100

Reasoning:
While the translation accurately preserves the core story context (the recipient, the care facility, and the location), significant flaws severely impact its fluency and readability:

1.  Catastrophic MT Hallucinations/Repetitions: The translation contains a major machine translation glitch (Dear Maine Dear Maine ’s Department of Health and Human ServicesServices - in Cincinnati), which severely damages readability and presentation quality.

2.  Mistranslation of Address Form: Leaving the salutation in English rather than integrating or translating it contextually breaks German text flow.

3.  Register/Grammar Inconsistency: "When you opened them" is literally translated using the formal second-person pronoun ("als Sie sie öffneten"), which creates an inappropriate shift in address and unnatural phrasing in German narrative prose.

In [ ]:
# Cell 5.5.2. MT Evaluation with CoT (Example 19, from Qian et al (2024), https://aclanthology.org/2024.emnlp-main.214/)

language_pair = "English-Dutch"
source_language = "English"
target_language = "Dutch"

source_sentence = "One bright morning in the Chicago Lying-in Hospital, a man named Edward K. Wehling, Jr., waited for his wife to give birth."
target_sentence = "Op een stralende ochtend in het Chicago Lying-in Hospital wachtte een man genaamd Edward K. Wehling Jr. op de geboorte van zijn vrouw."
reference_translation = "Op een stralende ochtend zat er een man genaamd Edward K. Wehling jr. in de kraamkliniek van Chicago te wachten tot zijn vrouw zou bevallen."

print("--- Automatic MT Evaluation with CoT ---")

# ============================================================
# PROMPT 1: Analyze the machine translation
# ============================================================

mt_eval_prompt1 = f"""Prompt 1: You are going to evaluate the quality of a {language_pair} translation. You need to think step by step. First, read the following source, machine translation and reference translation. Analyse where the machine translation is different from the reference translation.

Source: {source_sentence}

Machine translation: {target_sentence}

Reference translation: {reference_translation}
"""

print("\n--- Prompt 1: Translation Analysis ---")
print(mt_eval_prompt1)

# Call the LLM for Prompt 1
output_from_Prompt1 = generate_response(
    mt_eval_prompt1,
    temperature=0.2
)

print("\n--- Output from Prompt 1 ---")
print(output_from_Prompt1)


# ============================================================
# PROMPT 2: Score the machine translation
# ============================================================

mt_eval_prompt2 = f"""Prompt 2: A large language model did an evaluation of machine translation quality for the {source_language} sentence, which is given as below:

{output_from_Prompt1}

Based on your analysis, score the machine translation quality on a continuous scale from 0 to 100, where a score of zero means "no meaning preserved" and a score of one hundred means "perfect meaning and grammar". Provide the score strictly in JSON format.
"""

print("\n--- Prompt 2: MT Quality Scoring ---")
print(mt_eval_prompt2)

# Call the LLM for Prompt 2
output_from_Prompt2 = generate_response(
    mt_eval_prompt2,
    temperature=0.2
)

print("\n--- Output from Prompt 2 ---")
print(output_from_Prompt2)

--- Automatic MT Evaluation with CoT ---

--- Prompt 1: Translation Analysis ---
Prompt 1: You are going to evaluate the quality of a English-Dutch translation. You need to think step by step. First, read the following source, machine translation and reference translation. Analyse where the machine translation is different from the reference translation.

Source: One bright morning in the Chicago Lying-in Hospital, a man named Edward K. Wehling, Jr., waited for his wife to give birth.

Machine translation: Op een stralende ochtend in het Chicago Lying-in Hospital wachtte een man genaamd Edward K. Wehling Jr. op de geboorte van zijn vrouw.

Reference translation: Op een stralende ochtend zat er een man genaamd Edward K. Wehling jr. in de kraamkliniek van Chicago te wachten tot zijn vrouw zou bevallen.


--- Output from Prompt 1 ---
Here is a thinking step by step:

Here is a thinking step by step:

Here is a thinking step by step:

Here is a thinking step by step, much like the way to g

[link text](https://)Sample output --- Automatic MT Evaluation with CoT ---

Key differences:
- **Location phrasing**: Machine keeps the literal English "Chicago Lying-in Hospital"; reference translates to "kraamkliniek van Chicago" (maternity clinic of Chicago). The machine translation keeps the English proper noun phrase "Chicago Lying-in Hospital" intact, which sounds less natural in Dutch.
- Verb structure: Machine: "wachtte ... op de geboorte van zijn vrouw" (waited for the birth of his wife). Reference: "zat er een man ... in de kraamkliniek van Chicago te wachten tot zijn vrouw zou bevallen" (there sat a man ... in the maternities clinic of Chicago waiting until his wife would give birth). The reference implies the waiting is contingent on the birth happening, while the machine just says he waited for the birth of his wife. Both are valid, but the reference is more idiomatic Dutch.

Overall analysis of the translation quality: The machine translation is understandable but literal; the reference translation is more natural and idiomatic in Dutch, properly restructuring the location and verb phrasing to match Dutch idioms while keeping the core meaning intact. The capitalization of "Wehling" is a minor formatting difference.

--- Prompt 2: MT Quality Scoring ---
Prompt 2: A large language model did an evaluation of machine translation quality for the English sentence, which is given as below:

Based on your analysis, score the machine translation quality on a continuous scale from 0 to 100, where a score of zero means "no meaning preserved" and a score of one hundred means "perfect meaning and grammar". Provide the score strictly in JSON format.


--- Output from Prompt 2 ---
{"score": 78}

In [ ]:
# Cell 5.5.3. Linguistic QA (Example 20)

source_language = "English"
target_language = "Spanish"
customer_style = "NA"

lqa_prompt = f"""#Context#
You are a professional reviewer at a Language Service Provider.

#Objective#
Evaluate the quality of the translation in {target_language} translated from {source_language}.

#Style#
This is a technical document, following the style of {customer_style}.

#Tone#
Informative.

#Audience#
The translation is intended for a final user of this technology.

#Response#
Provide a markdown table where each row represents a sentence in the original translation:
- Column A: Source sentence
- Column B: Evaluated translation sentence
- Column C: Error category and explanation
- Column D: Suggested correction

Source text: "The foundation provides a seamless natural look."
Text for evaluation: "La fundación provee una mirada natural sin costuras."
"""

print(generate_response(lqa_prompt, temperature=0.1))

| A | B | C | D |
|---|---|---|---|
| The foundation provides a seamless natural look. | La fundación provee una mirada natural sin costuras. | **Semantic error**: "look" (appearance/finish) is mistranslated as "mirada" (gaze/glance). In a technical description of a product's appearance, "mirada" is inappropriate as it refers to a person's visual direction. The intended meaning requires "apariencia" or a retained "look." "Sin costuras" correctly renders "seamless," but "mirada natural" creates a meaning shift. | La fundación brinda una **apariencia** natural sin costuras. / La fundación provee un **look** natural sin costuras.


Sample output with Gemini 3.6 Flash

| Source sentence | Evaluated translation sentence | Error category and explanation | Suggested correction |
| --- | --- | --- | --- |
| The foundation provides a seamless natural look. | La fundación provee una mirada natural sin costuras. | **Mistranslation / Terminology:** <br>

<br>1. *“Fundación”* (organization/charity) is a false friend for cosmetic *foundation* (*base de maquillaje*).<br>

<br>2. *“Mirada”* (look/gaze) mistranslates *look* in the context of skin appearance (*aspecto* or *acabado*).<br>

<br>3. *“Sin costuras”* is a literal translation of *seamless* that applies to clothing, not cosmetics (*uniforme*, *imperceptible*, or *sin cortes*). | La base de maquillaje ofrece un aspecto natural e imperceptible. |

# 6. Prompt Engineering Tracker

As a final optional advanced exercise, one could document the iterative prompt adjustments, parameter configurations, and evaluation results. The code below is a starting point to implement this functionality.

In [ ]:
# Cell 6. Iteration Logging Tool
prompt_log = []

def log_experiment(version: str, task: str, prompt_text: str, temp: float, notes: str):
    """
    Appends a prompt trial entry to an in-memory execution table.
    """
    prompt_log.append({
        "Version": version,
        "Task": task,
        "Temperature": temp,
        "Prompt": prompt_text[:80] + "...",
        "Notes/Evaluation": notes
    })
    return pd.DataFrame(prompt_log)

# Log initial trial
log_experiment("v1.0", "Terminology Extraction", term_prompt, 0.1, "Extracted terms successfully; check URL links manually.")
log_experiment("v1.1", "Creative Translation", creative_prompt, 1.0, "High fluency, but check for minor additions.")

# Display log table
pd.DataFrame(prompt_log)